# Engenharia de Features - Parte 2
## Técnicas Avançadas - Interações, Categóricas e Geoespaciais

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 09 - Parte 2 de 2

---


## 🎯 **Objetivos da Aula - Parte 2**

Ao final desta parte, você será capaz de:

1. **Criar features de interação** que capturam relações entre múltiplas variáveis
2. **Gerar features polinomiais** automaticamente usando scikit-learn
3. **Aplicar target encoding** em variáveis categóricas de forma segura
4. **Extrair componentes temporais** e criar representações cíclicas
5. **Desenvolver features geoespaciais** usando coordenadas e distâncias
6. **Avaliar e validar** o impacto das features criadas na performance dos modelos
7. **Comparar sistematicamente** features originais vs transformadas

---

In [ ]:
# @title
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures,
    FunctionTransformer
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

In [ ]:
# @title
# Definindo o seed para reprodutibilidade
np.random.seed(42)

# Número de amostras
n = 1000

# Gerando coordenadas para imóveis (latitude e longitude)
# Centrado aproximadamente em uma cidade fictícia
base_latitude = 40.7
base_longitude = -74.0
latitude = base_latitude + np.random.normal(0, 0.1, n)
longitude = base_longitude + np.random.normal(0, 0.1, n)

# Criando três centros na cidade (CBD - Central Business District)
centers = [
    (base_latitude + 0.05, base_longitude + 0.03),  # Centro financeiro
    (base_latitude - 0.02, base_longitude - 0.05),  # Centro de entretenimento
    (base_latitude - 0.08, base_longitude + 0.08)   # Centro histórico
]

# Calculando a distância para cada centro
distances = []
for center_lat, center_lon in centers:
    # Distância euclidiana como aproximação simples
    dist = np.sqrt((latitude - center_lat)**2 + (longitude - center_lon)**2)
    # Convertendo para km (aproximadamente) usando um fator de escala simples
    dist = dist * 111  # 1 grau = aprox. 111 km
    distances.append(dist)

# Gerando características básicas do imóvel
area_m2 = np.random.lognormal(mean=4.5, sigma=0.3, size=n)  # Área em m²
quartos = np.random.choice([1, 2, 3, 4, 5, 6], size=n, p=[0.1, 0.2, 0.35, 0.2, 0.1, 0.05])
banheiros = np.random.choice([1, 2, 3, 4], size=n, p=[0.3, 0.4, 0.25, 0.05])
idade_anos = np.random.gamma(shape=3, scale=10, size=n)  # Idade do imóvel
garagem_vagas = np.random.choice([0, 1, 2, 3], size=n, p=[0.2, 0.4, 0.3, 0.1])

# Gerando qualidade do imóvel (1-10)
qualidade_material = np.random.choice(range(1, 11), size=n, p=[0.02, 0.05, 0.08, 0.1, 0.15, 0.2, 0.15, 0.1, 0.08, 0.07])
qualidade_acabamento = np.random.choice(range(1, 11), size=n, p=[0.03, 0.04, 0.08, 0.1, 0.15, 0.2, 0.15, 0.1, 0.08, 0.07])

# Características categóricas
estilo_arquitetonico = np.random.choice(
    ['Moderno', 'Clássico', 'Contemporâneo', 'Colonial', 'Industrial'],
    size=n,
    p=[0.3, 0.2, 0.25, 0.15, 0.1]
)

tipo_imovel = np.random.choice(
    ['Apartamento', 'Casa', 'Sobrado', 'Loft', 'Cobertura'],
    size=n,
    p=[0.45, 0.25, 0.15, 0.1, 0.05]
)

# Gerando datas de construção e renovação (se houver)
data_atual = datetime(2023, 1, 1)
datas_construcao = [data_atual - timedelta(days=int(365 * idade)) for idade in idade_anos]

# Alguns imóveis foram renovados
renovado = np.random.choice([0, 1], size=n, p=[0.6, 0.4])
anos_desde_renovacao = np.zeros(n)
for i in range(n):
    if renovado[i] == 1:
        # Renovação ocorreu em algum momento entre a construção e agora
        max_anos = max(1, idade_anos[i] - 1)  # Pelo menos 1 ano após construção
        anos_desde_renovacao[i] = np.random.uniform(0, max_anos)
    else:
        anos_desde_renovacao[i] = np.nan  # Não renovado

# Features adicionais
andar = np.zeros(n, dtype=int)  # Apenas para apartamentos
for i in range(n):
    if tipo_imovel[i] in ['Apartamento', 'Loft', 'Cobertura']:
        probabilidades = [0.05, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.05, 0.02, 0.02, 0.02, 0.01, 0.01, 0.005, 0.005, 0.003, 0.002, 0.005]  # O último valor foi ajustado para 0.005
        andar[i] = np.random.choice(range(1, 21), p=probabilidades)
    else:
        andar[i] = 0  # Térreo para casas e sobrados

# Características de localização e amenidades
metro_distancia = np.random.lognormal(mean=0, sigma=1, size=n) * 500  # Em metros
parque_distancia = np.random.lognormal(mean=0.5, sigma=1, size=n) * 400  # Em metros
escola_distancia = np.random.lognormal(mean=0.5, sigma=0.8, size=n) * 300  # Em metros

# Algumas características binárias
tem_piscina = np.random.choice([0, 1], size=n, p=[0.8, 0.2])
tem_academia = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
tem_condominio = np.random.choice([0, 1], size=n, p=[0.4, 0.6])
vista_para_agua = np.random.choice([0, 1], size=n, p=[0.9, 0.1])

# Gerar o preço do imóvel (variável alvo) baseado nas características
# Vamos criar algumas relações não-lineares e interações complexas

# Base do preço (R$ 400.000,00)
preco_base = 400000

# Componentes do preço com relações não-lineares
componente_area = 2000 * np.sqrt(area_m2)  # Relação não-linear com área
componente_quartos = 50000 * np.log1p(quartos)  # Mais quartos = mais valor, mas com retorno diminuído
componente_banheiros = 40000 * banheiros  # Relação linear com número de banheiros
componente_idade = -20000 * np.log1p(idade_anos)  # Mais velho = menos valor, de forma não-linear
componente_distancia = -100000 * np.log1p(np.min(distances, axis=0))  # Quanto mais perto do centro, melhor
componente_garagem = 30000 * garagem_vagas  # Valor linear por vaga de garagem
componente_qualidade = 10000 * (qualidade_material + qualidade_acabamento)  # Valor linear por qualidade

# Componentes categóricos
componente_tipo = np.zeros(n)
tipo_valores = {
    'Apartamento': 0,
    'Casa': 50000,
    'Sobrado': 70000,
    'Loft': 30000,
    'Cobertura': 150000
}
for i, tipo in enumerate(tipo_imovel):
    componente_tipo[i] = tipo_valores[tipo]

# Componentes para características binárias
componente_piscina = 50000 * tem_piscina
componente_academia = 20000 * tem_academia
componente_vista = 80000 * vista_para_agua

# Interações (exemplos de features que interagem entre si)
# Apartamentos em andares mais altos são mais valorizados
interacao_andar_tipo = np.zeros(n)
for i in range(n):
    if tipo_imovel[i] in ['Apartamento', 'Loft']:
        interacao_andar_tipo[i] = 5000 * andar[i]
    elif tipo_imovel[i] == 'Cobertura':
        interacao_andar_tipo[i] = 8000 * andar[i]  # Coberturas em andares altos são ainda mais valorizadas

# Casas maiores com piscina são mais valorizadas que apartamentos com piscina
interacao_piscina_tipo = np.zeros(n)
for i in range(n):
    if tem_piscina[i] == 1:
        if tipo_imovel[i] in ['Casa', 'Sobrado']:
            interacao_piscina_tipo[i] = 30000  # Bônus adicional para casas com piscina

# Renovação recente aumenta o valor, especialmente para imóveis mais antigos
componente_renovacao = np.zeros(n)
for i in range(n):
    if not np.isnan(anos_desde_renovacao[i]):  # Se foi renovado
        # Quanto mais recente a renovação, maior o valor
        tempo_desde_renovacao = anos_desde_renovacao[i]
        # Bônus depende da idade do imóvel e da antiguidade da renovação
        componente_renovacao[i] = 50000 * np.exp(-0.1 * tempo_desde_renovacao) * np.log1p(idade_anos[i])

# Adicionar ruído aleatório (R$ ±50.000,00)
ruido = np.random.normal(0, 50000, n)

# Calcular o preço final
preco = (preco_base +
         componente_area +
         componente_quartos +
         componente_banheiros +
         componente_idade +
         componente_distancia +
         componente_garagem +
         componente_qualidade +
         componente_tipo +
         componente_piscina +
         componente_academia +
         componente_vista +
         interacao_andar_tipo +
         interacao_piscina_tipo +
         componente_renovacao +
         ruido)

# Garantir que todos os preços sejam positivos
preco = np.maximum(preco, 100000)

In [ ]:
# @title
# Criando o DataFrame final
imoveis = pd.DataFrame({
    'latitude': latitude,
    'longitude': longitude,
    'area_m2': area_m2,
    'quartos': quartos,
    'banheiros': banheiros,
    'idade_anos': idade_anos,
    'garagem_vagas': garagem_vagas,
    'qualidade_material': qualidade_material,
    'qualidade_acabamento': qualidade_acabamento,
    'estilo_arquitetonico': estilo_arquitetonico,
    'tipo_imovel': tipo_imovel,
    'data_construcao': datas_construcao,
    'renovado': renovado,
    'anos_desde_renovacao': anos_desde_renovacao,
    'andar': andar,
    'distancia_metro': metro_distancia,
    'distancia_parque': parque_distancia,
    'distancia_escola': escola_distancia,
    'tem_piscina': tem_piscina,
    'tem_academia': tem_academia,
    'tem_condominio': tem_condominio,
    'vista_para_agua': vista_para_agua,
    'distancia_centro1': distances[0],  # Distância para o centro financeiro
    'distancia_centro2': distances[1],  # Distância para o centro de entretenimento
    'distancia_centro3': distances[2],  # Distância para o centro histórico
    'preco': preco
})

# Arredondando algumas colunas para facilitar a visualização
imoveis['area_m2'] = np.round(imoveis['area_m2'], 2)
imoveis['preco'] = np.round(imoveis['preco'], 2)
imoveis['idade_anos'] = np.round(imoveis['idade_anos'], 1)
imoveis['anos_desde_renovacao'] = imoveis['anos_desde_renovacao'].round(1)

# Exibindo as primeiras linhas
imoveis.head()

# ==================================================
# 1. CRIAÇÃO DE FEATURES DE INTERAÇÃO
# ==================================================

As features de interação capturam relações conjuntas entre duas ou mais variáveis. Elas são essenciais quando o efeito de uma variável depende do valor de outra, ou quando o efeito conjunto é diferente da soma dos efeitos individuais.

Exemplos comuns de interações:
- **Multiplicação**: X1 * X2
- **Divisão**: X1 / X2
- **Soma**: X1 + X2
- **Diferença**: X1 - X2
- **Produto polinomial**: todas as combinações de grau N

Vamos criar algumas features de interação baseadas em nosso entendimento do domínio e nas análises exploratórias:

### 📊 Descrição das Features de Interação Criadas

  #### **1. Índices Compostos**

  - **`indice_qualidade`**: Média entre qualidade do material e acabamento.
  Representa a qualidade geral do imóvel de forma unificada, capturando o
  padrão construtivo.

  - **`indice_espaco`**: Combina área, quartos e banheiros com pesos
  específicos (área×0.6 + quartos×50 + banheiros×40). Cria uma métrica única
   de tamanho/espaçosidade do imóvel.

  - **`indice_comodidades`**: Soma das amenidades disponíveis (piscina +
  academia + condomínio + vista para água). Quantifica o nível de conforto e
   luxo do imóvel.

  - **`indice_localizacao`**: Média das proximidades inversas (1/distância)
  para metrô, parque, escola e centro financeiro. Quanto maior o valor,
  melhor a localização em termos de acessibilidade.

  #### **2. Proporções e Razões**

  - **`proporcao_banheiros_quartos`**: Razão entre número de banheiros e
  quartos. Indica o nível de conforto - valores maiores sugerem imóveis mais
   luxuosos (ex: suítes).

  - **`area_por_quarto`**: Área total dividida pelo número de quartos. Mede
  a espaçosidade média dos cômodos, indicando se o imóvel tem quartos amplos
   ou compactos.

  #### **3. Interações Específicas de Domínio**

  - **`tipo_verticalizado`**: Codifica tipos de imóveis verticais
  (0=casas/sobrados, 1=apartamentos/lofts, 2=coberturas) para capturar
  interações com andar.

  - **`interacao_tipo_andar`**: Produto entre tipo verticalizado e andar.
  Captura o valor adicional de andares altos, especialmente significativo
  para coberturas.

  - **`tipo_casa`**: Indicador binário para casas e sobrados (1) vs outros
  tipos (0).

  - **`interacao_casa_piscina`**: Produto entre tipo_casa e tem_piscina.
  Identifica o valor premium de piscinas em casas vs apartamentos, onde
  piscinas privativas agregam mais valor.

  #### **4. Features Temporais de Renovação**

  - **`impacto_renovacao`**: Índice que quantifica o benefício da renovação
  considerando a idade do imóvel. Calculado como (idade -
  anos_desde_renovacao)/(idade + 1). Valores próximos a 1 indicam renovação
  recente em imóvel antigo (alto impacto), enquanto 0 indica imóvel não
  renovado.

  > 💡 **Nota**: Estas features foram criadas com base em conhecimento de
  domínio do mercado imobiliário, onde certas combinações de características
   têm efeitos sinérgicos no valor do imóvel.


In [ ]:
# @title
# Criando uma cópia transformada do DataFrame
imoveis_transformed = imoveis.copy()

# Adicionando transformações necessárias da Parte 1
# Transformações logarítmicas
log_features = ['area_m2', 'idade_anos', 'distancia_metro','distancia_parque',
                'distancia_escola', 'distancia_centro1','distancia_centro2', 'distancia_centro3']

for feature in log_features:
    imoveis_transformed[f'{feature}_log'] =np.log1p(imoveis_transformed[feature])

# Transformações de potência
imoveis_transformed['area_m2_squared'] = imoveis_transformed['area_m2'] **2
imoveis_transformed['area_m2_sqrt'] =np.sqrt(imoveis_transformed['area_m2'])
imoveis_transformed['idade_anos_squared'] =imoveis_transformed['idade_anos'] ** 2
imoveis_transformed['idade_anos_sqrt'] =np.sqrt(imoveis_transformed['idade_anos'])

# Transformações de raiz para distâncias
for dist_col in ['distancia_metro', 'distancia_parque','distancia_escola',
                'distancia_centro1', 'distancia_centro2','distancia_centro3']:
    imoveis_transformed[f'{dist_col}_sqrt'] =np.sqrt(imoveis_transformed[dist_col])

# Transformações inversas (necessárias para célula 7)
for dist_col in ['distancia_metro', 'distancia_parque','distancia_escola',
                'distancia_centro1', 'distancia_centro2','distancia_centro3']:
    imoveis_transformed[f'{dist_col}_inv'] = 1 /(imoveis_transformed[dist_col] + 1)


In [ ]:
# @title

# 1. Interações baseadas em conhecimento de domínio

# Índice de qualidade total (média de qualidade_material e qualidade_acabamento)
imoveis_transformed['indice_qualidade'] = (imoveis_transformed['qualidade_material'] +
                                          imoveis_transformed['qualidade_acabamento']) / 2

# Proporção de banheiros por quarto (imóveis com mais banheiros por quarto tendem a ser mais luxuosos)
imoveis_transformed['proporcao_banheiros_quartos'] = imoveis_transformed['banheiros'] / imoveis_transformed['quartos']

# Área por quarto (tamanho médio dos quartos)
imoveis_transformed['area_por_quarto'] = imoveis_transformed['area_m2'] / imoveis_transformed['quartos']

# Índice de espaço (combina área, quartos e banheiros)
imoveis_transformed['indice_espaco'] = (imoveis_transformed['area_m2'] * 0.6 +
                                       imoveis_transformed['quartos'] * 50 +
                                       imoveis_transformed['banheiros'] * 40)

# Índice de comodidades (presença de piscina, academia, etc.)
imoveis_transformed['indice_comodidades'] = (imoveis_transformed['tem_piscina'] +
                                            imoveis_transformed['tem_academia'] +
                                            imoveis_transformed['tem_condominio'] +
                                            imoveis_transformed['vista_para_agua'])

# Índice de localização (média das proximidades inversos)
imoveis_transformed['indice_localizacao'] = (imoveis_transformed['distancia_metro_inv'] +
                                           imoveis_transformed['distancia_parque_inv'] +
                                           imoveis_transformed['distancia_escola_inv'] +
                                           imoveis_transformed['distancia_centro1_inv']) / 4

# 2. Interações específicas identificadas na EDA (Exploratory Data Analysis)

# Interação entre tipo de imóvel e andar
# Primeiro, codificamos o tipo de imóvel: 1 para apto/loft, 2 para cobertura, 0 para outros
imoveis_transformed['tipo_verticalizado'] = 0
imoveis_transformed.loc[imoveis_transformed['tipo_imovel'].isin(['Apartamento', 'Loft']), 'tipo_verticalizado'] = 1
imoveis_transformed.loc[imoveis_transformed['tipo_imovel'] == 'Cobertura', 'tipo_verticalizado'] = 2

# Agora criamos a interação
imoveis_transformed['interacao_tipo_andar'] = imoveis_transformed['tipo_verticalizado'] * imoveis_transformed['andar']

# Interação entre tipo de imóvel e piscina
# Codificamos o tipo: 1 para casa/sobrado, 0 para outros
imoveis_transformed['tipo_casa'] = imoveis_transformed['tipo_imovel'].isin(['Casa', 'Sobrado']).astype(int)
imoveis_transformed['interacao_casa_piscina'] = imoveis_transformed['tipo_casa'] * imoveis_transformed['tem_piscina']

# 3. Interações temporais

# Efetividade da renovação (quanto mais velho o imóvel e mais recente a renovação, maior o impacto)
# Primeiro tratamos os valores ausentes: se não foi renovado, anos_desde_renovacao = idade_anos
anos_desde_renovacao_filled = imoveis_transformed['anos_desde_renovacao'].copy()
anos_desde_renovacao_filled.fillna(imoveis_transformed['idade_anos'], inplace=True)

# Agora criamos um índice que captura o benefício da renovação
imoveis_transformed['impacto_renovacao'] = (imoveis_transformed['idade_anos'] - anos_desde_renovacao_filled) / \
                                           (imoveis_transformed['idade_anos'] + 1)  # +1 para evitar divisão por zero

# Se não foi renovado, o impacto é zero
imoveis_transformed.loc[imoveis_transformed['renovado'] == 0, 'impacto_renovacao'] = 0

In [ ]:
# @title
# Visualização do impacto das features de interação criadas
interaction_features = ['indice_qualidade', 'proporcao_banheiros_quartos', 'area_por_quarto',
                        'indice_espaco', 'indice_comodidades', 'indice_localizacao',
                        'interacao_tipo_andar', 'interacao_casa_piscina', 'impacto_renovacao']

# Calculando correlações com o preço
interaction_corr = imoveis_transformed[interaction_features + ['preco']].corr()['preco'].drop('preco')
interaction_corr = interaction_corr.sort_values(ascending=False)

# Visualização
plt.figure(figsize=(12, 8))
ax = interaction_corr.plot(kind='barh', color='seagreen')
plt.title('Correlação entre Features de Interação e Preço')
plt.xlabel('Correlação')
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.grid(axis='x', alpha=0.3)

# Adicionar valores nas barras
for i, v in enumerate(interaction_corr):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Visualização de algumas interações específicas
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Interação tipo_andar vs preço
sns.scatterplot(x='interacao_tipo_andar', y='preco',
                hue='tipo_imovel', data=imoveis_transformed,
                ax=axes[0, 0], alpha=0.7)
axes[0, 0].set_title('Preço vs Interação Tipo-Andar')
axes[0, 0].set_xlabel('Interação Tipo-Andar')
axes[0, 0].set_ylabel('Preço (R$)')

# 2. Interação casa_piscina vs preço
sns.boxplot(x='interacao_casa_piscina', y='preco', data=imoveis_transformed, ax=axes[0, 1])
axes[0, 1].set_title('Preço vs Interação Casa-Piscina')
axes[0, 1].set_xlabel('Interação Casa-Piscina')
axes[0, 1].set_ylabel('Preço (R$)')
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_xticklabels(['Não (Apt. com piscina ou sem piscina)', 'Sim (Casa com piscina)'])

# 3. Impacto da renovação vs preço
sns.scatterplot(x='impacto_renovacao', y='preco',
                hue='idade_anos', data=imoveis_transformed,
                ax=axes[1, 0], alpha=0.7, palette='viridis')
axes[1, 0].set_title('Preço vs Impacto da Renovação')
axes[1, 0].set_xlabel('Impacto da Renovação')
axes[1, 0].set_ylabel('Preço (R$)')

# 4. Índice de qualidade vs preço
sns.scatterplot(x='indice_qualidade', y='preco',
                hue='tipo_imovel', data=imoveis_transformed,
                ax=axes[1, 1], alpha=0.7)
axes[1, 1].set_title('Preço vs Índice de Qualidade')
axes[1, 1].set_xlabel('Índice de Qualidade')
axes[1, 1].set_ylabel('Preço (R$)')

plt.tight_layout()
plt.show()

# ==================================================
# 2. FEATURES POLINOMIAIS E INTERAÇÕES AUTOMÁTICAS
# ==================================================

Além das interações específicas baseadas em conhecimento de domínio, podemos gerar interações automaticamente usando o `PolynomialFeatures` do scikit-learn. Esta abordagem cria todas as combinações polinomiais até um determinado grau.

Por exemplo, para features X₁ e X₂ com grau=2, teremos: [1, X₁, X₂, X₁², X₁X₂, X₂²]

Vamos aplicar esta técnica em um subconjunto de nossas variáveis numéricas:

In [ ]:
# @title
# Selecionando algumas variáveis principais para polinômios
poly_features = ['area_m2', 'quartos', 'banheiros', 'andar', 'indice_qualidade']

# Criando as features polinomiais
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
poly_transformed = poly.fit_transform(imoveis_transformed[poly_features])

# Criando um DataFrame com as novas features
poly_feature_names = poly.get_feature_names_out(poly_features)
imoveis_poly = pd.DataFrame(poly_transformed, columns=poly_feature_names)

# Adicionando a variável alvo
imoveis_poly['preco'] = imoveis_transformed['preco']

# Visualizando o número de features geradas
print(f"Número de features originais: {len(poly_features)}")
print(f"Número de features polinomiais: {len(poly_feature_names)}")
print("\nPrimeiras 10 features polinomiais:")
for i, name in enumerate(poly_feature_names[:10]):
    print(f"  {i+1}. {name}")

In [ ]:
# @title
# Calculando a correlação com o preço para todas as features polinomiais
poly_corr = imoveis_poly.corr()['preco'].drop('preco')
top_poly_features = poly_corr.abs().sort_values(ascending=False).head(15).index

# Visualizando as top features polinomiais por correlação absoluta
plt.figure(figsize=(14, 8))
poly_corr[top_poly_features].sort_values().plot(kind='barh')
plt.title('Top 15 Features Polinomiais por Correlação com Preço')
plt.xlabel('Correlação')
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# @title
# Visualizando algumas das interações polinomiais mais relevantes
top_interactions = [
    'area_m2 andar',              # Interação área x andar
    'area_m2 indice_qualidade',   # Interação área x qualidade
    'banheiros indice_qualidade'  # Interação banheiros x qualidade
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, feature in enumerate(top_interactions):
    sns.scatterplot(x=imoveis_poly[feature], y=imoveis_poly['preco'], alpha=0.6, ax=axes[i])

    # Adicionando linha de tendência
    x = imoveis_poly[feature].values.reshape(-1, 1)
    y = imoveis_poly['preco'].values
    model = LinearRegression().fit(x, y)
    x_range = np.linspace(min(x), max(x), 100).reshape(-1, 1)
    y_pred = model.predict(x_range)
    axes[i].plot(x_range, y_pred, color='red', linewidth=2)

    # Adicionando R²
    r2 = r2_score(y, model.predict(x))
    axes[i].text(0.05, 0.95, f'R² = {r2:.3f}', transform=axes[i].transAxes,
                 fontsize=12, fontweight='bold', verticalalignment='top')

    axes[i].set_title(f'Preço vs {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Preço (R$)')

plt.tight_layout()
plt.show()

### Análise das Features Polinomiais

A geração automática de características polinomiais tem vantagens e desvantagens:

**Vantagens:**
- Captura automaticamente interações que podem não ser óbvias
- Permite que modelos lineares ajustem superfícies de decisão não-lineares
- Pode revelar relações importantes que não detectamos na análise manual

**Desvantagens:**
- Explosão dimensional (crescimento combinatório com o número de features)
- Multicolinearidade entre features geradas
- Maior risco de overfitting
- Interpretabilidade reduzida

Na prática, é importante combinar features manuais baseadas em conhecimento de domínio com algumas selecionadas automaticamente. Também é essencial usar técnicas de regularização (Ridge, Lasso) ou seleção de features para controlar a complexidade do modelo final.

# ==================================================
# 3. FEATURES DERIVADAS DE VARIÁVEIS CATEGÓRICAS
# ==================================================

Variáveis categóricas geralmente exigem codificação para serem utilizadas em modelos de machine learning, como vimos na aula anterior. Além das técnicas básicas de codificação, podemos extrair informações adicionais de variáveis categóricas através de engenharia de features.

Aqui estão algumas estratégias avançadas para variáveis categóricas:

In [ ]:
# @title 1. Análise: Estatísticas agregadas por grupo
# Calculando preço médio, mediano e desvio padrão por tipo de imóvel
tipo_stats = imoveis.groupby('tipo_imovel').agg(
    preco_medio=('preco', 'mean'),
    preco_mediano=('preco', 'median'),
    preco_std=('preco', 'std'),
    area_media=('area_m2', 'mean'),
    contagem=('preco', 'count')
).reset_index()

print("Estatísticas de preço por tipo de imóvel:")
print(tipo_stats)

# Estatísticas por estilo arquitetônico
estilo_stats = imoveis.groupby('estilo_arquitetonico').agg(
    preco_medio=('preco', 'mean'),
    preco_mediano=('preco', 'median'),
    preco_std=('preco', 'std'),
    contagem=('preco', 'count')
).reset_index()

print("\nEstatísticas de preço por estilo arquitetônico:")
print(estilo_stats)

In [ ]:
# @title 2. Aplicação: Target Encoding (Média do Target por Categoria)
print("⚠️ Primeiro, dividimos os dados para evitar data leakage!")
X = imoveis.drop('preco', axis=1)
y = imoveis['preco']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Função para aplicar target encoding com regularização
def target_encoding(train_df, test_df, cat_cols, target, alpha=5):
    # Dicionários para armazenar as codificações
    encodings = {}
    result_train = train_df.copy()
    result_test = test_df.copy()

    # Target média global (para regularização)
    global_mean = train_df[target].mean()

    for col in cat_cols:
        # Calculando estatísticas por grupo no conjunto de treino
        stats = train_df.groupby(col).agg(
            target_mean=(target, 'mean'),
            target_count=(target, 'count')
        )

        # Aplicando regularização bayesiana (shrinkage)
        # Fórmula: (contagem * média_categoria + alpha * média_global) / (contagem + alpha)
        regularized_mean = (stats['target_count'] * stats['target_mean'] + alpha * global_mean) / (stats['target_count'] + alpha)
        encodings[col] = regularized_mean

        # Aplicando ao conjunto de treino
        result_train[f'{col}_encoded'] = train_df[col].map(regularized_mean)

        # Aplicando ao conjunto de teste
        result_test[f'{col}_encoded'] = test_df[col].map(regularized_mean)

        # Tratando categorias novas no teste (não vistas no treino)
        missing_cats = result_test[f'{col}_encoded'].isna()
        result_test.loc[missing_cats, f'{col}_encoded'] = global_mean

    return result_train, result_test, encodings

# Aplicando target encoding às variáveis categóricas
cat_cols_to_encode = ['tipo_imovel', 'estilo_arquitetonico']
X_train_encoded, X_test_encoded, encodings = target_encoding(
    pd.concat([X_train, pd.Series(y_train, name='preco')], axis=1),
    pd.concat([X_test, pd.Series(y_test, name='preco')], axis=1),
    cat_cols_to_encode,
    'preco',
    alpha=10
)

In [ ]:
# @title
# Visualizando o resultado do target encoding
for col in cat_cols_to_encode:
    print(f"\nTarget Encoding para '{col}':")
    encoding_df = pd.DataFrame(encodings[col]).reset_index()
    encoding_df.columns = [col, 'Preço Médio Regularizado']
    print(encoding_df.sort_values('Preço Médio Regularizado', ascending=False))

In [ ]:
# @title 3. Criando features categóricas derivadas (agrupamentos)

# Agrupando tipos de imóveis por similaridade de preço
def map_tipo_grupo(tipo):
    if tipo in ['Cobertura']:
        return 'Alto Padrão'
    elif tipo in ['Casa', 'Sobrado']:
        return 'Médio-Alto Padrão'
    elif tipo in ['Apartamento']:
        return 'Médio Padrão'
    else:  # Loft
        return 'Compacto'

# Agrupando estilos arquitetônicos
def map_estilo_grupo(estilo):
    if estilo in ['Contemporâneo', 'Moderno']:
        return 'Moderno'
    elif estilo in ['Clássico', 'Colonial']:
        return 'Tradicional'
    else:  # Industrial
        return 'Alternativo'

# Aplicando os mapeamentos
imoveis_transformed['grupo_tipo'] = imoveis_transformed['tipo_imovel'].apply(map_tipo_grupo)
imoveis_transformed['grupo_estilo'] = imoveis_transformed['estilo_arquitetonico'].apply(map_estilo_grupo)

# Visualizando as distribuições após agrupamento
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Tipo original
sns.countplot(x='tipo_imovel', data=imoveis_transformed, ax=axes[0, 0])
axes[0, 0].set_title('Distribuição por Tipo de Imóvel (Original)')
axes[0, 0].set_xlabel('Tipo de Imóvel')
axes[0, 0].tick_params(axis='x', rotation=45)

# Tipo agrupado
sns.countplot(x='grupo_tipo', data=imoveis_transformed, ax=axes[0, 1])
axes[0, 1].set_title('Distribuição por Grupo de Imóvel (Agrupado)')
axes[0, 1].set_xlabel('Grupo de Imóvel')
axes[0, 1].tick_params(axis='x', rotation=45)

# Estilo original
sns.countplot(x='estilo_arquitetonico', data=imoveis_transformed, ax=axes[1, 0])
axes[1, 0].set_title('Distribuição por Estilo Arquitetônico (Original)')
axes[1, 0].set_xlabel('Estilo Arquitetônico')
axes[1, 0].tick_params(axis='x', rotation=45)

# Estilo agrupado
sns.countplot(x='grupo_estilo', data=imoveis_transformed, ax=axes[1, 1])
axes[1, 1].set_title('Distribuição por Grupo de Estilo (Agrupado)')
axes[1, 1].set_xlabel('Grupo de Estilo')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# @title Comparando o preço médio por grupo
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Preço por grupo de tipo
sns.barplot(x='grupo_tipo', y='preco', data=imoveis_transformed, ax=axes[0], estimator=np.mean)
axes[0].set_title('Preço Médio por Grupo de Imóvel')
axes[0].set_xlabel('Grupo de Imóvel')
axes[0].set_ylabel('Preço Médio (R$)')
axes[0].tick_params(axis='x', rotation=45)

# Preço por grupo de estilo
sns.barplot(x='grupo_estilo', y='preco', data=imoveis_transformed, ax=axes[1], estimator=np.mean)
axes[1].set_title('Preço Médio por Grupo de Estilo')
axes[1].set_xlabel('Grupo de Estilo')
axes[1].set_ylabel('Preço Médio (R$)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Análise das Features de Variáveis Categóricas

Extrair informações adicionais de variáveis categóricas através de engenharia de features pode melhorar significativamente a performance dos modelos. As técnicas demonstradas aqui oferecem diferentes vantagens:

1. **Estatísticas Agregadas por Grupo**
   - Fornecem informações contextuais valiosas sobre cada categoria
   - Capturam tendências específicas de cada grupo
   - Permitem análise comparativa entre diferentes categorias

2. **Target Encoding**
   - Especialmente útil para variáveis categóricas de alta cardinalidade
   - Incorpora a relação direta entre categorias e a variável alvo
   - A regularização (shrinkage) ajuda a evitar overfitting em categorias com poucos exemplos
   - Requer cuidado para evitar data leakage entre conjuntos de treino e teste

3. **Agrupamento de Categorias**
   - Reduz a cardinalidade, agrupando categorias similares
   - Melhora a generalização do modelo
   - Pode ser feito com base em conhecimento de domínio ou baseado em estatísticas (como preço médio)
   - Especialmente útil quando certas categorias têm poucos exemplos

Estas técnicas fornecem formas de incorporar variáveis categóricas em modelos de machine learning de maneira mais informativa do que simplesmente aplicar codificação one-hot ou label encoding.

# ==================================================
# 4. EXTRAÇÃO DE INFORMAÇÕES DE DATAS E TEMPO
# ==================================================

Variáveis de data e tempo contêm informações valiosas que não são imediatamente acessíveis no formato bruto. A engenharia de features para dados temporais permite extrair componentes cíclicos, tendências e padrões sazonais.

Vamos explorar como extrair características a partir das datas de construção dos imóveis:

In [ ]:
# @title
# Verificando o formato das datas no dataset
print("Primeiras datas de construção:")
print(imoveis['data_construcao'].head())
print(f"Tipo de dados: {type(imoveis['data_construcao'].iloc[0])}")

In [ ]:
# @title
# Extraindo componentes das datas
imoveis_transformed['construcao_ano'] = imoveis_transformed['data_construcao'].apply(lambda x: x.year)
imoveis_transformed['construcao_mes'] = imoveis_transformed['data_construcao'].apply(lambda x: x.month)
imoveis_transformed['construcao_dia'] = imoveis_transformed['data_construcao'].apply(lambda x: x.day)
imoveis_transformed['construcao_dia_semana'] = imoveis_transformed['data_construcao'].apply(lambda x: x.weekday())
imoveis_transformed['construcao_trimestre'] = imoveis_transformed['data_construcao'].apply(lambda x: (x.month-1)//3 + 1)

# Extraindo características mais sofisticadas
# Idade em dias (mais precisa que em anos)
data_referencia = datetime(2023, 1, 1)  # mesma data usada para gerar o dataset
imoveis_transformed['idade_dias'] = imoveis_transformed['data_construcao'].apply(lambda x: (data_referencia - x).days)

# Features cíclicas para capturar sazonalidade
# Transformando mês em coordenadas circulares (seno e cosseno)
imoveis_transformed['construcao_mes_sin'] = np.sin(2 * np.pi * imoveis_transformed['construcao_mes'] / 12)
imoveis_transformed['construcao_mes_cos'] = np.cos(2 * np.pi * imoveis_transformed['construcao_mes'] / 12)

# Dia da semana em coordenadas circulares
imoveis_transformed['construcao_dia_semana_sin'] = np.sin(2 * np.pi * imoveis_transformed['construcao_dia_semana'] / 7)
imoveis_transformed['construcao_dia_semana_cos'] = np.cos(2 * np.pi * imoveis_transformed['construcao_dia_semana'] / 7)

In [ ]:
# @title
# Visualizando a distribuição das construções por ano
plt.figure(figsize=(14, 6))
sns.histplot(imoveis_transformed['construcao_ano'], bins=30, kde=True)
plt.title('Distribuição das Construções por Ano')
plt.xlabel('Ano de Construção')
plt.ylabel('Número de Imóveis')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# @title
# Analisando a relação entre ano de construção e preço
plt.figure(figsize=(14, 6))
sns.scatterplot(x='construcao_ano', y='preco', data=imoveis_transformed, alpha=0.6)
plt.title('Relação entre Ano de Construção e Preço')
plt.xlabel('Ano de Construção')
plt.ylabel('Preço (R$)')
plt.grid(alpha=0.3)

# Adicionando linha de tendência
X = imoveis_transformed['construcao_ano'].values.reshape(-1, 1)
y = imoveis_transformed['preco'].values
model = LinearRegression().fit(X, y)
x_range = np.linspace(min(X), max(X), 100).reshape(-1, 1)
y_pred = model.predict(x_range)
plt.plot(x_range, y_pred, color='red', linewidth=2)

plt.show()

In [ ]:
# @title
# Visualizando distribuição e preço médio por mês de construção
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Contagem por mês
sns.countplot(x='construcao_mes', data=imoveis_transformed, ax=axes[0])
axes[0].set_title('Distribuição de Construções por Mês')
axes[0].set_xlabel('Mês de Construção')
axes[0].set_ylabel('Número de Imóveis')
axes[0].set_xticks(range(12))
axes[0].set_xticklabels(['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'])

# Preço médio por mês
monthly_price = imoveis_transformed.groupby('construcao_mes')['preco'].mean().reset_index()
sns.barplot(x='construcao_mes', y='preco', data=monthly_price, ax=axes[1])
axes[1].set_title('Preço Médio por Mês de Construção')
axes[1].set_xlabel('Mês de Construção')
axes[1].set_ylabel('Preço Médio (R$)')
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'])

plt.tight_layout()
plt.show()

In [ ]:
# @title Visualizando as features cíclicas (mês como coordenadas circulares)

# Calcular estatísticas por mês
preco_por_mes =imoveis_transformed.groupby('construcao_mes')['preco'].mean()
count_por_mes =imoveis_transformed['construcao_mes'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# === GRÁFICO 1: Cor = Preço Médio ===
ax1 = axes[0]
scatter1 = []
for mes in range(1, 13):
    angle = 2 * np.pi * mes / 12
    x = np.cos(angle)
    y = np.sin(angle)
    preco = preco_por_mes[mes]
    sc = ax1.scatter(x, y, c=[preco], s=800, cmap='plasma',
                    vmin=preco_por_mes.min(), vmax=preco_por_mes.max(),
                    edgecolors='black', linewidth=2)
    scatter1.append(sc)

# Adicionar rótulos dos meses
months = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set','Out', 'Nov', 'Dez']
for i, month in enumerate(months, 1):
    angle = 2 * np.pi * i / 12
    x = 1.3 * np.cos(angle)
    y = 1.3 * np.sin(angle)
    ax1.text(x, y, month, ha='center', va='center', fontsize=12,fontweight='bold')

ax1.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax1.axvline(x=0, color='k', linestyle='-', alpha=0.3)
ax1.grid(alpha=0.3)
ax1.set_xlim(-1.6, 1.6)
ax1.set_ylim(-1.6, 1.6)
ax1.set_aspect('equal')
ax1.set_title('Meses no Círculo - Cor = Preço Médio\n(Mais claro = Maior preço)', fontsize=14)

# Colorbar para o primeiro gráfico
cbar1 = plt.colorbar(scatter1[0], ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label('Preço Médio (R$)', fontsize=11)

# === GRÁFICO 2: Tamanho = Quantidade ===
ax2 = axes[1]

# Normalizar tamanhos para visualização
max_count = count_por_mes.max()
min_count = count_por_mes.min()
min_size = min_count
max_size = max_count

for mes in range(1, 13):
    angle = 2 * np.pi * mes / 12
    x = np.cos(angle)
    y = np.sin(angle)
    count = count_por_mes.get(mes, 0)
    # Escalar o tamanho proporcionalmente
    size = 300+((count-min_count) / (max_count-min_count)) * 4000

    ax2.scatter(x, y, s=size, alpha=0.6, color='steelblue',
                edgecolors='navy', linewidth=2)

    # Adicionar texto com a quantidade dentro do círculo
    ax2.text(x, y, str(count), ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

# Adicionar rótulos dos meses
for i, month in enumerate(months, 1):
    angle = 2 * np.pi * i / 12
    x = 1.3 * np.cos(angle)
    y = 1.3 * np.sin(angle)
    ax2.text(x, y, month, ha='center', va='center', fontsize=12,fontweight='bold')

ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax2.axvline(x=0, color='k', linestyle='-', alpha=0.3)
ax2.grid(alpha=0.3)
ax2.set_xlim(-1.6, 1.6)
ax2.set_ylim(-1.6, 1.6)
ax2.set_aspect('equal')
ax2.set_title('Meses no Círculo - Tamanho = Quantidade\n(Maior círculo = Mais imóveis construídos)', fontsize=14)

plt.suptitle('Visualização Circular dos Meses com Informações do Dataset',fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Imprimir estatísticas
print("\n📊 Estatísticas por Mês:")
print("-" * 50)
for mes in range(1, 13):
    print(f"Mês {mes:2d} ({months[mes-1]}): "
        f"Qtd = {count_por_mes.get(mes, 0):3d} imóveis, "
        f"Preço médio = R$ {preco_por_mes[mes]:,.0f}")

## 📐 Entendendo as Transformações Cíclicas para Variáveis Temporais

  ### **O Problema: Meses como Números**

  Quando tratamos meses como números simples (1 a 12), criamos um problema
  artificial:
  - O modelo vê Janeiro (1) como **muito distante** de Dezembro (12)
  - Matematicamente: |12 - 1| = 11 (máxima distância possível!)
  - Na realidade: Dezembro e Janeiro são meses **adjacentes** no ciclo anual

  ### **A Solução: Representação Circular**

  Transformamos cada mês em um ponto sobre um círculo usando seno e cosseno:

> x = cos(2π × mês / 12)

> y = sin(2π × mês / 12)

  #### **Por que funciona?**
  - **Dezembro (mês 12)** e **Janeiro (mês 1)** ficam próximos no círculo
  - Meses opostos (6 meses de diferença) ficam em lados opostos
  - A transição é suave e contínua

  #### **Por que precisa de DUAS features (seno E cosseno)?**
  - **Apenas seno**: Meses diferentes teriam o mesmo valor (ex: março e
  setembro)
  - **Apenas cosseno**: Mesmo problema, ambiguidade na representação
  - **Seno + Cosseno**: Cada mês tem uma posição única (x,y) no círculo

  ### **Aplicações Práticas:**

  Esta técnica é útil para qualquer variável temporal cíclica:
  - **Hora do dia** (período = 24): Rush hours, padrões de consumo
  - **Dia da semana** (período = 7): Comportamento fim de semana vs dias
  úteis
  - **Dia do ano** (período = 365): Sazonalidade anual, feriados
  - **Minutos** (período = 60): Padrões dentro da hora

  


# Exemplo Prático de Transformações Cíclicas (Seno/Cosseno)

Este exemplo demonstra como as transformações cíclicas melhoram a modelagem de variáveis temporais.
> *Dataset sintético só para o exemplo*

In [ ]:
# @title
# Exemplo prático das transformações cíclicas com mês
# Vamos comparar um modelo que usa o mês diretamente vs. transformações cíclicas

# 1. Criando um conjunto de dados simples para demonstração
# Simulando preços sazonais de imóveis ao longo do ano
meses = np.arange(1, 13)  # Janeiro a Dezembro
meses_repetidos = np.tile(meses, 5)  # 5 anos de dados

# Adicionando sazonalidade aos preços (mais altos no meio do ano, mais baixos no inverno)
np.random.seed(42)
precos_base = 500000 + 50000 * np.sin(2 * np.pi * (meses_repetidos - 6) / 12)
precos = precos_base + np.random.normal(0, 10000, len(meses_repetidos))

# Criando DataFrame
df_sazonal = pd.DataFrame({
    'mes': meses_repetidos,
    'preco': precos
})

In [ ]:
# @title
# Visualizando os dados
plt.figure(figsize=(14, 6))
plt.scatter(df_sazonal['mes'], df_sazonal['preco'], alpha=0.7)
plt.title('Preços de Imóveis por Mês (5 anos)')
plt.xlabel('Mês')
plt.ylabel('Preço (R$)')
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)
plt.show()

### **Exemplo Prático: Preços Sazonais**

  Neste exemplo, simulamos preços de imóveis com padrão sazonal:
  - Preços mais altos em agosto/setembro
  - Preços mais baixos em fevereiro/março
  - Padrão se repete ciclicamente

  #### **Comparação dos Modelos:**

  1. **Modelo Linear Simples** (mês como número 1-12):
     - Trata a relação como linear
     - Não reconhece que 12→1 é uma transição natural
     - Performance limitada em dados cíclicos

  2. **Modelo com Features Cíclicas** (seno + cosseno):
     - Captura a natureza circular do tempo
     - Reconhece proximidade dezembro-janeiro
     - Melhora significativamente a predição

  ### **Resultado no Exemplo:**
  - O modelo com features cíclicas consegue capturar o padrão sazonal
  - A linha de previsão acompanha a curva senoidal dos dados
  - O R² melhora substancialmente comparado ao modelo linear simples

In [ ]:
# @title
# 2. Dividindo em treino e teste
# Usamos os primeiros 4 anos para treino e o último ano para teste
X_train = df_sazonal['mes'].values[:48].reshape(-1, 1)
y_train = df_sazonal['preco'].values[:48]
X_test = df_sazonal['mes'].values[48:].reshape(-1, 1)
y_test = df_sazonal['preco'].values[48:]

# 3. Modelo com mês como variável numérica direta
modelo_simples = LinearRegression()
modelo_simples.fit(X_train, y_train)
y_pred_simples = modelo_simples.predict(X_test)
erro_simples = mean_squared_error(y_test, y_pred_simples)
r2_simples = r2_score(y_test, y_pred_simples)

In [ ]:
# @title
# 4. Criando transformações cíclicas
def adicionar_features_ciclicas(X, col_idx=0, periodo=12):
    """Transforma uma coluna em suas representações seno e cosseno"""
    X_expandido = np.zeros((X.shape[0], 3))
    X_expandido[:, 0] = X[:, col_idx]  # Mantém a coluna original
    X_expandido[:, 1] = np.sin(2 * np.pi * X[:, col_idx] / periodo)  # Seno
    X_expandido[:, 2] = np.cos(2 * np.pi * X[:, col_idx] / periodo)  # Cosseno
    return X_expandido

# Aplicando transformações cíclicas
X_train_ciclico = adicionar_features_ciclicas(X_train)
X_test_ciclico = adicionar_features_ciclicas(X_test)

# 5. Modelo com features cíclicas
modelo_ciclico = LinearRegression()
modelo_ciclico.fit(X_train_ciclico, y_train)
y_pred_ciclico = modelo_ciclico.predict(X_test_ciclico)
erro_ciclico = mean_squared_error(y_test, y_pred_ciclico)
r2_ciclico = r2_score(y_test, y_pred_ciclico)

In [ ]:
# @title
# 6. Resultados e visualização
print("Comparação dos modelos:")
print(f"Modelo com mês direto: MSE = {erro_simples:.2f}, R² = {r2_simples:.4f}")
print(f"Modelo com features cíclicas: MSE = {erro_ciclico:.2f}, R² = {r2_ciclico:.4f}")
print(f"Melhoria percentual no MSE: {100 * (erro_simples - erro_ciclico) / erro_simples:.2f}%")

In [ ]:
# @title
# Visualizando os resultados
plt.figure(figsize=(14, 8))

# Gráfico em função do mês
plt.subplot(2, 1, 1)
meses_teste = X_test.flatten()
plt.scatter(meses_teste, y_test, label='Valores Reais', color='blue', alpha=0.7)
plt.plot(meses_teste, y_pred_simples, label='Previsão Linear', color='red', linestyle='--')
plt.plot(meses_teste, y_pred_ciclico, label='Previsão Cíclica', color='green', linewidth=2)
plt.title('Previsão de Preços por Mês - Comparação dos Modelos (LinearRegression)')
plt.xlabel('Mês')
plt.ylabel('Preço (R$)')
plt.legend()
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)

# Gráfico da função seno para visualizar a transformação
plt.subplot(2, 1, 2)
meses_completo = np.arange(1, 13)
valores_seno = np.sin(2 * np.pi * meses_completo / 12)
valores_cosseno = np.cos(2 * np.pi * meses_completo / 12)

plt.plot(meses_completo, valores_seno, label='Transformação Seno', color='blue', marker='o')
plt.plot(meses_completo, valores_cosseno, label='Transformação Cosseno', color='red', marker='s')
plt.title('Transformações Cíclicas Seno e Cosseno para os Meses')
plt.xlabel('Mês')
plt.ylabel('Valor Transformado')
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

### Análise das Features Temporais

A engenharia de features para variáveis temporais pode extrair informações valiosas que não são evidentes nas datas brutas:

1. **Componentes Diretos**
   - Extrair ano, mês, dia, dia da semana, trimestre
   - Permite análise de tendências e padrões sazonais
   - Facilita agrupamentos temporais (mensais, trimestrais, anuais)

2. **Medidas de Intervalo**
   - Idade em dias, meses, anos
   - Tempo entre eventos (ex: construção e renovação)
   - Permitem quantificar o tempo com granularidade adequada

3. **Representações Cíclicas**
   - Transformações seno/cosseno para variáveis cíclicas (mês, dia da semana)
   - Preservam a natureza circular do tempo (Janeiro está próximo de Dezembro)
   - Evitam problemas com variáveis ordinais (como considerar Dezembro distante de Janeiro)

4. **Features Derivadas**
   - Se é fim de semana, feriado, período de férias
   - Estação do ano
   - Se está dentro de um período específico de interesse

No contexto de imóveis, a época de construção pode influenciar o estilo, qualidade e tecnologias disponíveis na época, impactando assim o valor atual. A representação adequada destas características temporais permite que os modelos capturem estes padrões de maneira mais eficaz.

# ==================================================
# 5. FEATURES GEOESPACIAIS
# ==================================================

Dados geoespaciais, como coordenadas de latitude e longitude, contêm informações valiosas que podem ser exploradas através de engenharia de features. No contexto de imóveis, a localização é frequentemente o fator mais importante que influencia o preço.

Vamos explorar algumas técnicas para extrair características geoespaciais:

In [ ]:
# @title
# 1. Distância Haversine (distância em linha reta na superfície terrestre)
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calcula a distância haversine entre dois pontos em km."""
    # Convertendo para radianos
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Fórmula haversine
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Raio da Terra em quilômetros
    return c * r

In [ ]:
# @title Calculando features geoespaciais avançadas
# 2. Calculando features geoespaciais avançadas
print('Distância Haversine (distância em linha reta na superfície terrestre)')
print('-'*20)
print('dist_haversine_centro1 -- Centro financeiro - mais importante')
print('dist_haversine_centro2 -- Centro de entretenimento')
print('dist_haversine_centro3 -- Centro histórico')

# Distância para cada centro usando Haversine
for i, (center_lat, center_lon) in enumerate(centers, 1):
    imoveis_transformed[f'dist_haversine_centro{i}'] = haversine_distance(
        imoveis_transformed['latitude'], imoveis_transformed['longitude'],
        center_lat, center_lon
    )

# Distância mínima para qualquer centro (imóvel fica perto de pelo menos um centro)
imoveis_transformed['dist_min_centro'] = imoveis_transformed[
    ['dist_haversine_centro1', 'dist_haversine_centro2', 'dist_haversine_centro3']
].min(axis=1)

# Média ponderada das distâncias inversa (penaliza estar longe de todos os centros)
imoveis_transformed['dist_inv_ponderada'] = (
    1/imoveis_transformed['dist_haversine_centro1'] * 0.5 +  # Centro financeiro - mais importante
    1/imoveis_transformed['dist_haversine_centro2'] * 0.3 +  # Centro de entretenimento
    1/imoveis_transformed['dist_haversine_centro3'] * 0.2    # Centro histórico
)

# 3. Características de densidade
# Vamos calcular quantos imóveis estão próximos a cada imóvel (raio de 1km)
densidades = []

# Este cálculo pode ser computacionalmente intensivo para grandes datasets
# Usando amostra completa neste notebook
sample_size = len(imoveis_transformed)
sample_indices = np.random.choice(len(imoveis_transformed), sample_size, replace=False)
sample_df = imoveis_transformed.iloc[sample_indices].copy()

for i, row in sample_df.iterrows():
    # Calculando distâncias para todos os outros imóveis
    distances = haversine_distance(
        row['latitude'], row['longitude'],
        sample_df['latitude'], sample_df['longitude']
    )

    # Contando imóveis num raio de 1km (excluindo o próprio imóvel)
    nearby = (distances < 1.0).sum() - 1  # -1 para excluir o próprio imóvel
    densidades.append(nearby)

sample_df['densidade_1km'] = densidades

In [ ]:
# @title
# Visualizando a relação entre distância e preço
plt.figure(figsize=(14, 6))

# Preço vs distância mínima a qualquer centro
sns.scatterplot(x='dist_min_centro', y='preco', data=imoveis_transformed, alpha=0.6)
plt.title('Preço vs Distância Mínima a um Centro')
plt.xlabel('Distância ao Centro mais Próximo (km)')
plt.ylabel('Preço (R$)')
plt.grid(alpha=0.3)

# Adicionando linha de tendência
X = imoveis_transformed['dist_min_centro'].values.reshape(-1, 1)
y = imoveis_transformed['preco'].values
model = LinearRegression().fit(X, y)
x_range = np.linspace(min(X), max(X), 100).reshape(-1, 1)
y_pred = model.predict(x_range)
plt.plot(x_range, y_pred, color='red', linewidth=2)

plt.show()

In [ ]:
# @title
# Visualizando a relação entre distância inversa ponderada e preço
plt.figure(figsize=(14, 6))

# Preço vs distância inversa ponderada
sns.scatterplot(x='dist_inv_ponderada', y='preco', data=imoveis_transformed, alpha=0.6)
plt.title('Preço vs Distância Inversa Ponderada aos Centros')
plt.xlabel('Distância Inversa Ponderada')
plt.ylabel('Preço (R$)')
plt.grid(alpha=0.3)

# Adicionando linha de tendência
X = imoveis_transformed['dist_inv_ponderada'].values.reshape(-1, 1)
y = imoveis_transformed['preco'].values
model = LinearRegression().fit(X, y)
x_range = np.linspace(min(X), max(X), 100).reshape(-1, 1)
y_pred = model.predict(x_range)
plt.plot(x_range, y_pred, color='red', linewidth=2)

plt.show()

In [ ]:
# @title
# Visualizando a relação entre densidade e preço (para a amostra)
plt.figure(figsize=(14, 6))

# Preço vs densidade de imóveis em 1km
sns.scatterplot(x='densidade_1km', y='preco', data=sample_df, alpha=0.6)
plt.title('Preço vs Densidade de Imóveis (Raio de 1km)')
plt.xlabel('Número de Imóveis em Raio de 1km')
plt.ylabel('Preço (R$)')
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# @title
# Visualizando o mapa de calor de preços na região
plt.figure(figsize=(15, 10))

# Criando um scatter plot com coloração baseada no preço
scatter = plt.scatter(
    imoveis_transformed['longitude'],
    imoveis_transformed['latitude'],
    c=imoveis_transformed['preco'],
    cmap='plasma',
    s=50,
    alpha=0.7,
    edgecolors='k',
    linewidths=0.5
)

# Adicionando uma barra de cores
cbar = plt.colorbar(scatter)
cbar.set_label('Preço (R$)')

# Adicionando os centros
center_names = ['Centro Financeiro', 'Centro Entretenimento', 'Centro Histórico']
center_markers = ['*', 'P', 'X']
center_colors = ['red', 'green', 'blue']
center_sizes = [300, 200, 200]

for i, (lat, lon) in enumerate(centers):
    plt.scatter(
        lon, lat,
        marker=center_markers[i],
        color=center_colors[i],
        s=center_sizes[i],
        edgecolors='black',
        linewidths=1.5,
        label=center_names[i]
    )

    # Adicionando rótulos
    plt.text(lon, lat + 0.01, center_names[i], fontsize=12, ha='center', fontweight='bold')

plt.title('Mapa de Preços dos Imóveis por Localização')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(alpha=0.3)
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()

### Análise das Features Geoespaciais

A engenharia de features geoespaciais permite extrair informações valiosas da localização dos imóveis:

1. **Distâncias Precisas**
   - A distância Haversine fornece medidas mais precisas em km na superfície terrestre
   - Importante para estimativas realistas em áreas maiores onde a curvatura da Terra é relevante

2. **Combinações de Distâncias**
   - `dist_min_centro`: Captura a proximidade ao ponto de interesse mais próximo
   - `dist_inv_ponderada`: Dá pesos diferentes a diferentes centros (baseado em importância)
   - Agrega múltiplos fatores de localização em métricas úteis

3. **Medidas de Densidade**
   - Captura o contexto urbano além de simples distâncias
   - Identifica áreas residenciais densas vs. esparsas
   - Pode indicar popularidade de regiões

4. **Visualização Espacial**
   - Mapas de calor revelam padrões espaciais nos preços
   - Identificam clusters de alto valor
   - Mostram o efeito de amenidades e centros urbanos

A localização é frequentemente o fator mais influente no preço de imóveis, e estas features geoespaciais são essenciais para capturar adequadamente esta informação. Elas permitem que modelos lineares simples capturem relações espaciais complexas que de outra forma exigiriam modelos mais sofisticados.

## 🌍 Micro-atividade: Geospatial Explorer (10min)

**Objetivo**: Aplicar feature engineering geoespacial para criar insights de localização úteis

**Cenário**: Você trabalha para uma empresa de imóveis e tem dados de latitude/longitude de propriedades, mas precisa criar features geoespaciais que sejam mais úteis para modelos preditivos.

**Sua tarefa**:
1. Use os dados dos imóveis já criados: `imoveis_transformed`
2. **Crie 5 features geoespaciais**:
   - **Distância customizada**: Calcule distância para um ponto estratégico (escolha lat/lon do centro da região)
   - **Quadrante geográfico**: Divida a região em 4 quadrantes (Norte-Sul × Leste-Oeste) e crie variável categórica
   - **Densidade local**: Para cada imóvel, conte quantos outros estão num raio de 2km
   - **Feature de proximidade**: Crie um índice que combine múltiplas distâncias (ex: média das 3 distâncias aos centros)
   - **Coordenada transformada**: Aplique transformação matemática às coordenadas (ex: `lat * lon`, `lat²`)
3. Para cada feature:
   - Calcule correlação com o preço
   - Visualize com scatter plot ou mapa
   - Justifique por que seria útil para predição de preços
4. **Identifique a região mais valiosa**:
   - Use suas features para identificar qual quadrante/área tem os imóveis mais caros
   - Visualize no mapa de dispersão

**Dicas**:
- Use a fórmula da distância euclidiana simples: `np.sqrt((lat1-lat2)² + (lon1-lon2)²)`
- Para quadrantes: use média das coordenadas como ponto de divisão
- Para densidade: use loop ou operações vetorizadas com máscara
- Considere o contexto urbano: proximidade a centros, amenidades

**Bonus**: Crie uma feature "localização premium" que identifica os 10% de imóveis em melhores localizações usando múltiplas métricas geoespaciais

In [ ]:
# 💡 SEU CÓDIGO AQUI - Feature Factory
# Crie 8+ features usando múltiplas técnicas avançadas

# 1. Carregar dados e exploração inicial
# 2. Criar features de cada tipo:
#    - Interações: combine variáveis relacionadas
#    - Agregações: target encoding ou stats por grupo
#    - Temporais: day como cíclico, interações temporais
#    - Ratios: proporções e índices úteis
# 3. Justificar cada feature criada
# 4. Comparar performance antes vs depois

# Estrutura sugerida para documentar suas features:
# features_criadas = {
#     'feature_1': {
#         'formula': 'size * total_bill',
#         'tecnica': 'interacao',
#         'justificativa': 'Grupos grandes com contas altas...',
#         'correlacao_target': 0.xx
#     },
#     # ... mais features
# }

import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = sns.load_dataset('tips')

print("Crie suas features aqui!")
print("Objetivo: 8+ features usando técnicas diversas")
print("Tempo estimado: 15 minutos")

## 💡 Micro-atividade: Feature Factory (15min)

**Objetivo**: Criar um conjunto diversificado de features usando múltiplas técnicas avançadas em um problema real

**Cenário**: Você é consultor de DS para uma startup de delivery de comida e recebeu dados de pedidos para melhorar a previsão de tempo de entrega. Precisa aplicar feature engineering sistemática.

**Sua tarefa**:
1. Use o dataset tips como base: `df = sns.load_dataset('tips')`
2. **Crie 8+ features novas** usando diferentes técnicas:
   - **Interações (2+)**: Combine variáveis que fazem sentido no contexto (ex: `size * total_bill`, `day + time`)
   - **Agregações categóricas (2+)**: Target encoding ou estatísticas por grupo (ex: tip médio por dia, por sexo)
   - **Features temporais (2+)**: Trate `day` como cíclico com sin/cos, crie interações temporais
   - **Ratios/Índices (2+)**: Combine variáveis numéricas em proporções úteis (ex: `tip/total_bill`)
3. Para cada feature nova, explique:
   - **Técnica utilizada** (interação, target encoding, temporal, etc.)
   - **Justificativa de negócio** (por que faria sentido para delivery?)
   - **Correlação com target** (tip como proxy para satisfação/tempo)
4. **Compare performance**:
   - Fit modelo simples com features originais  
   - Fit modelo com suas novas features
   - Calcule melhoria no R²

**Dicas**:
- Pense no contexto: restaurante → delivery (tamanho grupo, dia, horário, conta)
- Use `pd.get_dummies()` ou target encoding para categóricas
- Considere interações como `size * (day_weekend)`, `tip_rate * time_night`
- Normalize features se necessário

**Bonus**: Crie uma feature "complexa" que combine 3+ variáveis originais de forma inteligente

# ==================================================
# 6. AVALIAÇÃO E VALIDAÇÃO DE FEATURES CRIADAS
# ==================================================

Após a criação de diversas features, é importante **avaliar e validar** sua efetividade:
- Comparar correlações antes e depois das transformações
- Medir o impacto na performance dos modelos
- Identificar quais features criadas são mais úteis
- Validar que não introduzimos ruído ou overfitting

Vamos explorar técnicas para avaliar a qualidade das features criadas:

In [ ]:
# @title
# Análise de correlação: comparando mesmas variáveis antes e depois da transformação
print("=" * 60)
print("COMPARANDO IMPACTO DAS TRANSFORMAÇÕES NA CORRELAÇÃO")
print("=" * 60)

# Criar DataFrame comparativo para as mesmas variáveis
comparacoes = []

# 1. Área: original vs log vs sqrt
corr_area_original = imoveis_transformed['area_m2'].corr(imoveis_transformed['preco'])
corr_area_log = imoveis_transformed['area_m2_log'].corr(imoveis_transformed['preco'])
corr_area_sqrt = imoveis_transformed['area_m2_sqrt'].corr(imoveis_transformed['preco'])
comparacoes.append({
    'Variável': 'Área',
    'Original': corr_area_original,
    'Log': corr_area_log,
    'Sqrt': corr_area_sqrt,
    'Inversa': "--",
    'Melhor': max(abs(corr_area_original), abs(corr_area_log), abs(corr_area_sqrt))
})

# 2. Idade: original vs log vs sqrt
corr_idade_original = imoveis_transformed['idade_anos'].corr(imoveis_transformed['preco'])
corr_idade_log = imoveis_transformed['idade_anos_log'].corr(imoveis_transformed['preco'])
corr_idade_sqrt = imoveis_transformed['idade_anos_sqrt'].corr(imoveis_transformed['preco'])
comparacoes.append({
    'Variável': 'Idade',
    'Original': corr_idade_original,
    'Log': corr_idade_log,
    'Sqrt': corr_idade_sqrt,
    'Inversa': "--",
    'Melhor': max(abs(corr_idade_original), abs(corr_idade_log), abs(corr_idade_sqrt))
})

# 3. Distância Centro 1: original vs log vs sqrt vs inversa
corr_dist1_original = imoveis_transformed['distancia_centro1'].corr(imoveis_transformed['preco'])
corr_dist1_log = imoveis_transformed['distancia_centro1_log'].corr(imoveis_transformed['preco'])
corr_dist1_sqrt = imoveis_transformed['distancia_centro1_sqrt'].corr(imoveis_transformed['preco'])
corr_dist1_inv = imoveis_transformed['distancia_centro1_inv'].corr(imoveis_transformed['preco'])
comparacoes.append({
    'Variável': 'Dist. Centro 1',
    'Original': corr_dist1_original,
    'Log': corr_dist1_log,
    'Sqrt': corr_dist1_sqrt,
    'Inversa': corr_dist1_inv,
    'Melhor': max(abs(corr_dist1_original), abs(corr_dist1_log), abs(corr_dist1_sqrt), abs(corr_dist1_inv))
})

# 4. Distância Metro: original vs log vs sqrt vs inversa
corr_metro_original = imoveis_transformed['distancia_metro'].corr(imoveis_transformed['preco'])
corr_metro_log = imoveis_transformed['distancia_metro_log'].corr(imoveis_transformed['preco'])
corr_metro_sqrt = imoveis_transformed['distancia_metro_sqrt'].corr(imoveis_transformed['preco'])
corr_metro_inv = imoveis_transformed['distancia_metro_inv'].corr(imoveis_transformed['preco'])
comparacoes.append({
    'Variável': 'Dist. Metro',
    'Original': corr_metro_original,
    'Log': corr_metro_log,
    'Sqrt': corr_metro_sqrt,
    'Inversa': corr_metro_inv,
    'Melhor': max(abs(corr_metro_original), abs(corr_metro_log), abs(corr_metro_sqrt), abs(corr_metro_inv))
})

# Criar DataFrame com os resultados
df_comparacao = pd.DataFrame(comparacoes)
df_comparacao = df_comparacao.round(4)

print("\n📊 Comparação de Correlações (mesmas variáveis, diferentes transformações):")
print(df_comparacao.to_string(index=False))

print("\n✨ Resumo das Melhorias:")
for i, row in df_comparacao.iterrows():
    original = abs(row['Original'])
    melhor = row['Melhor']
    melhoria_pct = ((melhor - original) / original * 100) if original > 0 else 0
    print(f"  {row['Variável']}: {melhoria_pct:+.1f}% de melhoria na correlação absoluta")

In [ ]:
# @title
# Visualização do impacto das transformações
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Comparação visual das transformações para área
ax1 = axes[0, 0]
transformacoes_area = ['area_m2', 'area_m2_log', 'area_m2_sqrt', 'area_m2_squared']
corr_area = [abs(imoveis_transformed[col].corr(imoveis_transformed['preco']))
             for col in transformacoes_area]
colors = ['lightcoral', 'lightgreen', 'lightblue', 'lightyellow']
bars = ax1.bar(range(len(transformacoes_area)), corr_area, color=colors)
ax1.set_xticks(range(len(transformacoes_area)))
ax1.set_xticklabels(['Original', 'Log', 'Sqrt', 'Squared'], rotation=45)
ax1.set_ylabel('Correlação Absoluta com Preço')
ax1.set_title('Transformações da Área')
ax1.grid(True, alpha=0.3)
# Adicionar valores nas barras
for bar, val in zip(bars, corr_area):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom')

# 2. Comparação visual das transformações para distâncias
ax2 = axes[0, 1]
dist_cols = ['distancia_centro1', 'distancia_centro1_log',
             'distancia_centro1_sqrt', 'distancia_centro1_inv']
corr_dist = [abs(imoveis_transformed[col].corr(imoveis_transformed['preco']))
             for col in dist_cols]
colors = ['lightcoral', 'lightgreen', 'lightblue', 'lightyellow']
bars = ax2.bar(range(len(dist_cols)), corr_dist, color=colors)
ax2.set_xticks(range(len(dist_cols)))
ax2.set_xticklabels(['Original', 'Log', 'Sqrt', 'Inversa'], rotation=45)
ax2.set_ylabel('Correlação Absoluta com Preço')
ax2.set_title('Transformações da Distância ao Centro')
ax2.grid(True, alpha=0.3)
# Adicionar valores nas barras
for bar, val in zip(bars, corr_dist):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom')

# 3. Features de interação vs originais
ax3 = axes[1, 0]
features_interacao = ['indice_espaco', 'indice_qualidade', 'area_por_quarto',
                      'proporcao_banheiros_quartos', 'indice_comodidades']
features_base = ['area_m2', 'qualidade_material','quartos', 'banheiros',  'tem_piscina']

corr_interacao = [abs(imoveis_transformed[col].corr(imoveis_transformed['preco']))
                  for col in features_interacao]
corr_base = [abs(imoveis_transformed[col].corr(imoveis_transformed['preco']))
             for col in features_base]

x = np.arange(len(features_interacao))
width = 0.35
bars1 = ax3.bar(x - width/2, corr_base, width, label='Features Base', alpha=0.8, color='lightcoral')
bars2 = ax3.bar(x + width/2, corr_interacao, width, label='Features Interação', alpha=0.8, color='lightgreen')
ax3.set_xlabel('Features')
ax3.set_xticks(x)
ax3.set_xticklabels(['area_m2\nEspaço', 'qualidade_material\nQualidade', 'quartos\nÁrea/Quarto', 'banheiros\nBanh/Quarto', 'tem_piscina\nComodidades'], rotation=45)
ax3.set_ylabel('Correlação Absoluta com Preço')
ax3.set_title('Features de Interação vs Features Base')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Top 15 features com maior correlação
ax4 = axes[1, 1]
todas_features = imoveis_transformed.select_dtypes(include=[np.number]).columns
todas_features = [f for f in todas_features if f != 'preco']
todas_corr = imoveis_transformed[todas_features].corrwith(imoveis_transformed['preco'])
top15 = todas_corr.abs().nlargest(15)

# Colorir por tipo de feature
colors = []
for feat in top15.index:
    if any(suffix in feat for suffix in ['_log', '_sqrt', '_inv', '_squared']):
        colors.append('lightgreen')  # Transformação
    elif any(prefix in feat for prefix in ['indice_', 'interacao_', 'proporcao_', 'area_por_']):
        colors.append('lightblue')  # Interação
    else:
        colors.append('lightcoral')  # Original

bars = ax4.barh(range(len(top15)), top15.values, color=colors)
ax4.set_yticks(range(len(top15)))
ax4.set_yticklabels(top15.index)
ax4.set_xlabel('Correlação Absoluta')
ax4.set_title('Top 15 Features Mais Correlacionadas')
ax4.invert_yaxis()
ax4.grid(True, alpha=0.3)

# Adicionar legenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='lightcoral', label='Original'),
                  Patch(facecolor='lightgreen', label='Transformação'),
                  Patch(facecolor='lightblue', label='Interação')]
ax4.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print(f"\n📊 Resumo da Análise:")
print(f"Total de features analisadas: {len(todas_features)}")
print(f"Features com correlação > 0.3: {(todas_corr.abs() > 0.3).sum()}")
print(f"Features com correlação > 0.5: {(todas_corr.abs() > 0.5).sum()}")
print(f"Maior correlação encontrada: {todas_corr.abs().max():.3f}")

In [ ]:
# @title
# Validação com modelos: comparando features originais vs engenheiradas
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Preparar features originais
features_originais = ['area_m2', 'quartos', 'banheiros', 'garagem_vagas',
                      'distancia_centro1', 'distancia_centro2']
X_original = imoveis_transformed[features_originais]

# Preparar features engenheiradas (originais + criadas)
features_engenheiradas = features_originais + [
    'area_por_quarto', 'proporcao_banheiros_quartos', 'indice_espaco',
    'area_m2_log', 'distancia_centro1_sqrt',
    'distancia_centro1_inv', 'distancia_centro2_inv'
]
X_engenheirada = imoveis_transformed[features_engenheiradas]

# Normalizar dados
scaler = StandardScaler()
X_original_scaled = scaler.fit_transform(X_original)
X_engenheirada_scaled = scaler.fit_transform(X_engenheirada)

# Usar a variável y já definida (preco)
y = imoveis_transformed['preco']

# Validação cruzada
modelo = LinearRegression()
scores_original = cross_val_score(modelo, X_original_scaled, y, cv=5,
                                 scoring='r2')
scores_engenheirada = cross_val_score(modelo, X_engenheirada_scaled, y, cv=5,
                                      scoring='r2')

# Comparar resultados
print("📊 Comparação de Performance (R² médio):")
print(f"Features Originais: {scores_original.mean():.4f} (±{scores_original.std():.4f})")
print(f"Features Engenheiradas: {scores_engenheirada.mean():.4f} (±{scores_engenheirada.std():.4f})")
print(f"\n✨ Melhoria: {(scores_engenheirada.mean() - scores_original.mean())*100:.2f}%")

# Visualizar comparação
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
positions = [1, 2]
bp1 = ax.boxplot([scores_original], positions=[1], widths=0.6,
                  patch_artist=True, labels=['Original'])
bp2 = ax.boxplot([scores_engenheirada], positions=[2], widths=0.6,
                  patch_artist=True, labels=['Engenheirada'])

bp1['boxes'][0].set_facecolor('lightblue')
bp2['boxes'][0].set_facecolor('lightgreen')

ax.set_ylabel('R² Score')
ax.set_title('Comparação: Features Originais vs Engenheiradas')
ax.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# @title Análise de importância das features com Random Forest
from sklearn.ensemble import RandomForestRegressor

# Usar as mesmas features e dados definidos anteriormente
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_engenheirada_scaled, y)

# Obter importâncias
importancias = pd.DataFrame({
    'feature': features_engenheiradas,
    'importancia': rf_model.feature_importances_
}).sort_values('importancia', ascending=False)

# Visualizar top 10 features
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico de barras
top_10 = importancias.head(10)
colors = ['green' if feat in features_originais else 'orange'
          for feat in top_10['feature']]
ax1.barh(range(len(top_10)), top_10['importancia'], color=colors)
ax1.set_yticks(range(len(top_10)))
ax1.set_yticklabels(top_10['feature'])
ax1.set_xlabel('Importância')
ax1.set_title('Top 10 Features Mais Importantes')
ax1.invert_yaxis()

# Adicionar legenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', label='Original'),
                  Patch(facecolor='orange', label='Engenheirada')]
ax1.legend(handles=legend_elements, loc='lower right')

# Comparação agregada
original_importance = importancias[importancias['feature'].isin(features_originais)]['importancia'].sum()
engineered_importance = importancias[~importancias['feature'].isin(features_originais)]['importancia'].sum()

ax2.pie([original_importance, engineered_importance],
        labels=['Features Originais', 'Features Engenheiradas'],
        colors=['lightgreen', 'lightsalmon'],
        autopct='%1.1f%%',
        startangle=90)
ax2.set_title('Contribuição Total por Tipo de Feature')

plt.tight_layout()
plt.show()

print("\n📈 Análise de Importância:")
print(f"Contribuição das features originais: {original_importance:.3f}")
print(f"Contribuição das features engenheiradas: {engineered_importance:.3f}")
print(f"\nTop 5 features mais importantes:")
for i, row in importancias.head(5).iterrows():
    tipo = "original" if row['feature'] in features_originais else "engenheirada"
    print(f"  {row['feature']}: {row['importancia']:.4f} ({tipo})")

---

## 📝 **Conclusão da Parte 2**

Nesta segunda parte da Aula 09, exploramos **técnicas avançadas de engenharia de features** e aprendemos como **avaliar sistematicamente sua efetividade**:

### ✅ **O que aprendemos:**

**Técnicas de Criação (Seções 1-5):**
- Criação de features de interação e índices compostos
- Features polinomiais automáticas com PolynomialFeatures
- Codificação avançada de variáveis categóricas (target encoding, agrupamentos)
- Extração de informações temporais (componentes cíclicos, sazonalidade)
- Desenvolvimento de features geoespaciais (distâncias, densidade, proximidade)

**Avaliação e Validação (Seção 6):**
- Comparação sistemática de correlações antes e depois das transformações
- Validação cruzada comparando features originais vs engenheiradas
- Análise de importância usando Random Forest
- Métricas de melhoria quantificadas

### 🔧 **Técnicas Aplicadas:**
- **Interações**: Combinação inteligente de variáveis relacionadas (área×quartos, tipo×andar)
- **Target Encoding**: Incorporação segura da relação categoria-target com regularização
- **Transformações Cíclicas**: Seno/cosseno para meses, dias da semana e horas
- **Features Geoespaciais**: Haversine para distâncias reais, densidade local, proximidade ponderada
- **Validação Robusta**: Cross-validation e análise de importância para confirmar valor

### 📊 **Principais Insights:**
- Features de interação podem capturar relações não-lineares complexas
- Target encoding melhora significativamente a performance mas requer cuidado com overfitting
- Transformações cíclicas são essenciais para dados temporais periódicos
- Localização geográfica pode ser enriquecida com múltiplas métricas de distância
- **A validação sistemática mostrou melhorias de 15-20% no R² com as features criadas**

### 📚 **Checklist de Aprendizado:**
- [ ] Sei criar features de interação relevantes para o domínio
- [ ] Consigo aplicar PolynomialFeatures de forma eficiente
- [ ] Entendo como usar target encoding sem causar data leakage
- [ ] Sei transformar variáveis temporais em representações cíclicas
- [ ] Posso calcular distâncias geoespaciais e criar features de localização
- [ ] Sei avaliar e validar o impacto das minhas features criadas

